# 🚀 Phase 1: Foundation, Infrastructure & Literature Validation Pipeline
## *Task-Technology Fit Analysis of Modern AI-Driven Intrusion Detection (Q1 Publication Pipeline)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

---

### 📌 Notebook Objectives:
1. **Google Drive Setup & Auto-Resolution**: Establish persistent drive storage at `ROOT: /content/drive/My Drive/Colab Notebook/` (or `/content/My Drive/Colab Notebooks/`).
2. **Git Safeguards Execution**: Automatically update `.gitignore` so large dataset files (`*.csv`, `*.pcap`, `/data/`) are never committed.
3. **Literature Validation Audit**: Run OpenAlex & CrossRef metadata harvesting and verify active DOIs with 0 retractions for 35 citations.
4. **Real vs. Synthetic Data Ingestion**: Inspect Google Drive `data/raw/` for authentic datasets (`MachineLearningCVE`, `unsw-data-full`, `TON-IoT`, `CIC-DDoS2019`, `NSL-KDD`), execute decontamination, and output clear status badges.
5. **CLI & Python Module Execution**: Demonstrate both Python API imports and Colab terminal commands (`!python src/data/prep_pipeline.py`).


### 1. ☁️ Google Drive Mount & Project Root Auto-Resolution

This cell automatically mounts Google Drive and detects your repository location across both singular `Colab Notebook` and plural `Colab Notebooks` paths.


In [ ]:
import os, sys
from pathlib import Path

# 1. Mount Google Drive if running inside Colab
try:
    from google.colab import drive
    if not Path('/content/drive').exists() and not Path('/content/My Drive').exists():
        drive.mount('/content/drive')
except ImportError:
    print("ℹ️ Running in local/workstation environment.")

# 2. Candidate root paths (supporting both 'Colab Notebook' and 'Colab Notebooks')
CANDIDATE_ROOTS = [
    Path('/content/drive/My Drive/Colab Notebook'),
    Path('/content/drive/My Drive/Colab Notebooks'),
    Path('/content/drive/MyDrive/Colab Notebook'),
    Path('/content/drive/MyDrive/Colab Notebooks'),
    Path('/content/My Drive/Colab Notebook'),
    Path('/content/My Drive/Colab Notebooks'),
    Path('/content/Colab Notebook'),
    Path('/content/Colab Notebooks'),
    Path('/Colab Notebook'),
    Path('/Colab Notebooks'),
    Path('/content/drive/MyDrive/is_ai-vuln'),
    Path('/content/drive/My Drive/is_ai-vuln'),
    Path('/content/is_ai-vuln'),
    Path('.').resolve()
]

PROJECT_ROOT = None
for cand in CANDIDATE_ROOTS:
    if cand.exists() and ((cand / 'src').exists() or (cand / 'data' / 'raw').exists()):
        PROJECT_ROOT = cand.resolve()
        break

# Dynamic discovery inside /content/drive if not yet matched
if PROJECT_ROOT is None and Path('/content/drive').exists():
    for drive_parent in [Path('/content/drive/My Drive'), Path('/content/drive/MyDrive'), Path('/content/drive'), Path('/content/My Drive')]:
        if drive_parent.exists():
            try:
                for sub in drive_parent.iterdir():
                    if sub.is_dir() and ('colab notebook' in sub.name.lower() or 'is_ai-vuln' in sub.name.lower()):
                        if (sub / 'src').exists() or (sub / 'data' / 'raw').exists():
                            PROJECT_ROOT = sub.resolve()
                            break
            except Exception:
                pass
            if PROJECT_ROOT:
                break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path('.').resolve()

os.chdir(str(PROJECT_ROOT))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Inspect data/raw directory to confirm real datasets presence
raw_dir = PROJECT_ROOT / 'data' / 'raw'
detected_folders = []
if raw_dir.exists():
    try:
        detected_folders = [f.name for f in raw_dir.iterdir() if f.is_dir()]
    except Exception:
        pass

print("=" * 80)
print(f"✅ Active Project Root : {PROJECT_ROOT}")
print(f"✅ src/ directory found: {(PROJECT_ROOT / 'src').exists()}")
print(f"📁 Raw Data Path       : {raw_dir.resolve()}")
print(f"🔍 Detected Raw Folders: {detected_folders}")
known_real = ['CIC-DDoS2019', 'MachineLearningCVE', 'NSL-KDD', 'TON-IoT', 'ToN-IOT', 'TrafficLabelling', 'unsw-data-full']
found_known = [f for f in detected_folders if f in known_real]
if found_known:
    print(f"🛡️ [DATA STATUS: AUTHENTIC REAL DATASETS DETECTED] Found: {found_known}")
else:
    print("ℹ️ [DATA STATUS] Note: Real datasets will also be dynamically located across Drive search paths.")
print("=" * 80)


### 2. 📦 Core Dependencies Installation


In [ ]:
!pip install -q scikit-learn scipy pandas numpy matplotlib seaborn networkx requests tqdm imbalanced-learn
print("✅ Core dependencies installed successfully.")


### 3. 🔒 Git Safeguards & Data Directory Layout

Ensures that `.gitignore` prevents accidentally pushing multi-gigabyte PCAP/CSV files to git, and initializes:
- `ROOT/data/raw/` (place your real dataset folders here: `MachineLearningCVE`, `TON-IoT`, `unsw-data-full`, etc.)
- `ROOT/data/processed/` (decontaminated outputs and partitions)


In [ ]:
from src.utils.environment import setup_environment
from src.data.drive_downloader import ensure_gitignore_safeguards, initialize_dataset_directories

# 1. Update .gitignore with automated safeguards
ensure_gitignore_safeguards(PROJECT_ROOT)

# 2. Initialize persistent storage layout
dirs = initialize_dataset_directories(PROJECT_ROOT)
print(f"📁 Raw Data Path      : {dirs['raw'].resolve()}")
print(f"📁 Processed Data Path: {dirs['processed'].resolve()}")


### 4. 📚 35-Reference Literature Validation & DOI Integrity Audit

Validates academic rigor against OpenAlex and CrossRef databases, verifying that all 35 references have active DOIs and 0 retractions.


In [ ]:
from src.utils.references_harvester import harvest_and_verify_metadata
from src.utils.references_validator import validate_citations_integrity

bib_path = PROJECT_ROOT / "references" / "library.bib"
if bib_path.exists():
    report = harvest_and_verify_metadata(str(bib_path))
    print(f"📚 References Scanned: {report['summary']['total_references']}")
    print(f"✅ Active DOIs: {report['summary']['active_dois']}")
    print(f"🚫 Retractions: {report['summary']['retracted_count']}")
else:
    print(f"ℹ️ BibTeX library not found at: {bib_path}")


### 5. 🛡️ Data Ingestion & Decontamination (Real vs. Synthetic Detection)

> [!IMPORTANT]
> **Dataset Ingestion Protocol**:
> - **Real Data**: If you have uploaded dataset folders (e.g. `MachineLearningCVE`, `unsw-data-full`, `TON-IoT`, `CIC-DDoS2019`, `NSL-KDD`) into `ROOT/data/raw/`, the pipeline will automatically detect it and perform full decontamination on real traffic.
> - **Synthetic Fallback**: If files are missing, the pipeline will generate a 10,000-sample synthetic dataset for dry-run verification and display a prominent alert banner.


In [ ]:
from src.data.prep_pipeline import run_preparation_pipeline
from src.data.drive_downloader import initialize_dataset_directories

dirs = initialize_dataset_directories(PROJECT_ROOT)

# Run ingestion, decontamination & 5-fold cross-validation partition
# Set prefer_sample=False to prioritize authentic files in data/raw/
prep_result = run_preparation_pipeline(
    dataset_name="CICIDS2017",
    base_dir=PROJECT_ROOT,
    prefer_sample=False,
    n_splits=5,
    build_graphs=True
)

print("\n" + "=" * 80)
if prep_result["is_synthetic"]:
    print("⚠️ [DATA NOTICE: PIPELINE EXECUTED ON SYNTHETIC FALLBACK DATA]")
    print(f"   To run on real data, please place dataset folders in: {dirs['raw'].resolve()}")
else:
    print("🛡️ [DATA NOTICE: PIPELINE EXECUTED ON AUTHENTIC REAL DATASET]")
    print(f"📁 Source File  : {prep_result['raw_source_file']}")
    print(f"📊 Cleaned Shape: {prep_result['cleaned_shape']}")
    print(f"💾 Cleaned File : {prep_result['processed_file']}")
print("=" * 80)


### 6. 💻 Python CLI Script Execution

You can also run the data preparation pipeline directly from the command line using the Python CLI interface:


In [ ]:
# 1. View CLI available arguments
!python src/data/prep_pipeline.py --help

# 2. Run data preparation via CLI command
!python src/data/prep_pipeline.py --dataset CICIDS2017 --n-splits 5


### 7. 🔄 Fault-Tolerant Checkpoint & Hardware Management Verification


In [ ]:
from src.utils.checkpoint_manager import CheckpointManager
from src.utils.environment import flush_memory

chk_dir = PROJECT_ROOT / "checkpoints"
chk_dir.mkdir(parents=True, exist_ok=True)

manager = CheckpointManager(
    drive_checkpoint_dir=chk_dir,
    dataset_name="CICIDS2017",
    track_name="Phase1_Verification",
    total_folds=5
)

flush_memory()
print("✅ Phase 1 Pipeline verified and ready for Phase 2 Benchmarks.")
